# Realistic people images — free on Kaggle or Colab

Two free, open models for photorealistic **fictional** people:

| Model | Strength | Speed on a free T4 | Needs |
|---|---|---|---|
| **RealVisXL V5.0** (SDXL) | portraits, fashion, lifestyle photos | ~15–25 s per image | nothing |
| **FLUX.1-schnell** (4-bit) | the most natural skin, hands, lighting | ~40–90 s per image | a free Hugging Face token |

Licences: RealVisXL is CreativeML OpenRAIL++-M, FLUX.1-schnell is Apache-2.0. Both allow commercial use.

**Use it for fictional people only.** Don't make images that look like real, identifiable people (celebrities, people you know) without their consent, or fake "real" photos meant to deceive anyone.

### Run on Kaggle (30 free GPU hours a week)
1. kaggle.com → **Create → New Notebook → File → Import Notebook**, and upload this file.
2. Right panel → **Session options**: Accelerator **GPU T4 x2**, **Internet on** (internet needs a phone-verified account).
3. For FLUX: **Add-ons → Secrets → Add secret** named `HF_TOKEN` (see the token steps below) and tick it for this notebook.
4. **Run All**. Images land in `/kaggle/working/people`. Download `people.zip` from the Output panel.

### Run on Colab
1. colab.research.google.com → **File → Upload notebook**.
2. **Runtime → Change runtime type → T4 GPU**.
3. For FLUX: the key icon (Secrets) → add `HF_TOKEN`, turn on notebook access.
4. **Runtime → Run all**. Download `people.zip` from the Files panel.

### Free Hugging Face token (for FLUX only)
1. Sign in at huggingface.co and open https://huggingface.co/black-forest-labs/FLUX.1-schnell, then click **Agree and access** (approved instantly).
2. https://huggingface.co/settings/tokens → **Create new token** → type **Read** → copy it into the `HF_TOKEN` secret.

In [ ]:
# 1. Install the image libraries (versions pinned so the notebook keeps working).
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "diffusers==0.41.0", "transformers==5.19.0", "accelerate==1.15.0",
                "bitsandbytes==0.50.2", "sentencepiece", "protobuf"], check=True)

In [ ]:
# 2. What to make. Edit the prompts, then run everything below.
import os

# Describe a photo, not a painting: camera, lens, light, skin and fabric detail.
PROMPTS = [
    "candid photo of a fictional Indian woman, late 20s, in a dim edit suite, monitor glow on her face, "
    "natural skin pores, amber light, 85mm f/1.8, photorealistic",
    "street portrait of a fictional man in his 40s, salt-and-pepper beard, rainy Mumbai night, "
    "wet neon reflections, 35mm documentary photo, sharp eyes",
    "studio headshot of a fictional young East African woman, short natural hair, soft window light, "
    "grey backdrop, 100mm, realistic skin texture",
]
# Keep each prompt short (about 40 words): RealVisXL reads only the first 77 tokens.
# Things to steer away from (RealVisXL only; FLUX.1-schnell ignores negative prompts).
NEGATIVE = ("cartoon, anime, illustration, painting, 3d render, cgi, plastic skin, airbrushed, deformed hands, "
            "extra fingers, blurry, low quality, watermark, text")

IMAGES_PER_PROMPT = 2      # per model
SEED = 1234                # same seed + prompt = same image; change it for new variations
RUN_REALVIS = True
RUN_FLUX = True            # needs the HF_TOKEN secret (see the top of the notebook)

OUT_DIR = ("/kaggle/working/people" if os.path.isdir("/kaggle/working")
           else "/content/people" if os.path.isdir("/content") else "people")
os.makedirs(OUT_DIR, exist_ok=True)
print("Saving to", OUT_DIR)

In [ ]:
# 3. Helpers: GPU check, token lookup, memory cleanup.
import gc, torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU found. On Kaggle/Colab switch the accelerator to a T4 GPU, or this will take hours.")

def hf_token():
    # Kaggle secret → Colab secret → environment variable.
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        pass
    return os.environ.get("HF_TOKEN")

def free_memory():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

def generator(seed):
    return torch.Generator(device="cpu").manual_seed(seed)

In [ ]:
# 4. RealVisXL V5.0: photoreal SDXL fine-tune, no token needed.
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

REALVIS_ID = "SG161222/RealVisXL_V5.0"

def load_realvis(model_id=REALVIS_ID):
    gpu = DEVICE == "cuda"
    pipe = StableDiffusionXLPipeline.from_pretrained(
        model_id, torch_dtype=torch.float16 if gpu else torch.float32,
        use_safetensors=True, **({"variant": "fp16"} if gpu else {}))
    # DPM++ 2M Karras: the sampler RealVisXL is tuned for.
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(
        pipe.scheduler.config, algorithm_type="dpmsolver++", use_karras_sigmas=True)
    return pipe.to(DEVICE)

def run_realvis(pipe, prompts, n, width=896, height=1152, steps=30, guidance=5.0):
    paths = []
    for i, prompt in enumerate(prompts):
        for k in range(n):
            image = pipe(prompt=prompt, negative_prompt=NEGATIVE, width=width, height=height,
                         num_inference_steps=steps, guidance_scale=guidance,
                         generator=generator(SEED + i * 100 + k)).images[0]
            path = f"{OUT_DIR}/realvis_{i + 1:02d}_{k + 1}.png"
            image.save(path)
            paths.append(path)
            print("saved", path)
    return paths

realvis_paths = []
if RUN_REALVIS:
    pipe = load_realvis()
    realvis_paths = run_realvis(pipe, PROMPTS, IMAGES_PER_PROMPT)
    del pipe
    free_memory()

In [ ]:
# 5. FLUX.1-schnell, 4-bit so it fits a free 16 GB T4. Needs the HF_TOKEN secret.
from diffusers import FluxPipeline, FluxTransformer2DModel, BitsAndBytesConfig as DiffusersBnb
from transformers import T5EncoderModel, BitsAndBytesConfig as TransformersBnb

FLUX_ID = "black-forest-labs/FLUX.1-schnell"

def load_flux(model_id=FLUX_ID, token=None, quantize=True):
    gpu = DEVICE == "cuda"
    # T4s have no native bfloat16; Ampere and newer (A100, L4) do.
    dtype = (torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16) if gpu else torch.float32
    parts = {}
    if quantize and gpu:
        # 4-bit transformer (~7 GB) + 8-bit T5 text encoder (~5 GB) instead of ~33 GB at full size.
        parts["transformer"] = FluxTransformer2DModel.from_pretrained(
            model_id, subfolder="transformer", torch_dtype=dtype, token=token,
            quantization_config=DiffusersBnb(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                             bnb_4bit_compute_dtype=dtype))
        parts["text_encoder_2"] = T5EncoderModel.from_pretrained(
            model_id, subfolder="text_encoder_2", torch_dtype=dtype, token=token,
            quantization_config=TransformersBnb(load_in_8bit=True))
    pipe = FluxPipeline.from_pretrained(model_id, torch_dtype=dtype, token=token, **parts)
    return pipe.to(DEVICE)

def run_flux(pipe, prompts, n, width=896, height=1152, steps=4):
    paths = []
    for i, prompt in enumerate(prompts):
        for k in range(n):
            # schnell is distilled for ~4 steps with no guidance.
            image = pipe(prompt=prompt, width=width, height=height, num_inference_steps=steps,
                         guidance_scale=0.0, max_sequence_length=256,
                         generator=generator(SEED + i * 100 + k)).images[0]
            path = f"{OUT_DIR}/flux_{i + 1:02d}_{k + 1}.png"
            image.save(path)
            paths.append(path)
            print("saved", path)
    return paths

flux_paths = []
if RUN_FLUX:
    token = hf_token()
    if not token:
        print("Skipping FLUX: add an HF_TOKEN secret (see the top of the notebook).")
    else:
        try:
            pipe = load_flux(token=token)
            flux_paths = run_flux(pipe, PROMPTS, IMAGES_PER_PROMPT)
            del pipe
            free_memory()
        except OSError as e:
            print("Couldn't download FLUX. Did you click 'Agree and access' on the model page "
                  "with the same account as the token?\n", e)

In [ ]:
# 6. Contact sheet + one zip to download.
import math, shutil
from PIL import Image

paths = realvis_paths + flux_paths
if paths:
    thumbs = [Image.open(p).convert("RGB") for p in paths]
    w = 320
    thumbs = [t.resize((w, round(t.height * w / t.width))) for t in thumbs]
    cols = min(4, len(thumbs))
    rows = math.ceil(len(thumbs) / cols)
    h = max(t.height for t in thumbs)
    sheet = Image.new("RGB", (cols * w, rows * h), "black")
    for j, t in enumerate(thumbs):
        sheet.paste(t, ((j % cols) * w, (j // cols) * h))
    sheet.save(f"{OUT_DIR}/contact_sheet.jpg", quality=90)
    archive = shutil.make_archive(os.path.join(os.path.dirname(os.path.abspath(OUT_DIR)), "people"), "zip", OUT_DIR)
    print(f"{len(paths)} images. Download: {archive}")
    try:
        from IPython.display import display
        display(sheet)
    except Exception:
        pass
else:
    print("No images were made.")